### 베이지안 최적화 개요와 HyperOpt 사용법

In [64]:
import hyperopt

print(hyperopt.__version__)

0.2.7


In [65]:
from hyperopt import hp

# search space (검색 공간) 설정

# -10 ~ 10까지 1간격을 가지는 입력 변수 x 집합값과 -15 ~ 15까지 1간격을 가지는 입력 변수 y 집합값 설정
search_space = {'x' : hp.quniform('x',-10, 10, 1), 'y' : hp.quniform('y', -15, 15, 1)}


In [66]:
search_space

{'x': <hyperopt.pyll.base.Apply at 0x13959c1d0>,
 'y': <hyperopt.pyll.base.Apply at 0x139581ad0>}

In [67]:
from hyperopt import STATUS_OK

# 목적 함수를 생성, 입력 변수값과 입력 변수 검색 범위를 가지는 딕셔너리를 인자로 받고, 특정 값을 반환합니다.

def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    
    retval = x**2 - 20*y
    
    return retval # {'loss' : retval, 'status' : STATUS_OK }

In [68]:
from hyperopt import fmin, tpe, Trials
import numpy as np

# 입력 결괏값을 저장한 Trials 객체값 생성
trial_val = Trials()

# 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 5번의 입력값 시도(max_evals=5)로 찾아냄
best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5,
               trials=trial_val, rstate=np.random.default_rng(seed=0))

print("best : ", best_01)


100%|██████████| 5/5 [00:00<00:00, 1895.99trial/s, best loss: -224.0]
best :  {'x': np.float64(-4.0), 'y': np.float64(12.0)}


In [69]:
trial_val = Trials()

# 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 20번의 입력값 시도(max_evals=20)로 찾아냄
best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20,
               trials=trial_val, rstate=np.random.default_rng(seed=0))

print("best : ", best_02)

100%|██████████| 20/20 [00:00<00:00, 2910.79trial/s, best loss: -296.0]
best :  {'x': np.float64(2.0), 'y': np.float64(15.0)}


**HyperOpt 수행 시 적용된 입력 값들과 목적 함수 반환 값 보기**

In [70]:
trial_val.results

[{'loss': -64.0, 'status': 'ok'},
 {'loss': -184.0, 'status': 'ok'},
 {'loss': 56.0, 'status': 'ok'},
 {'loss': -224.0, 'status': 'ok'},
 {'loss': 61.0, 'status': 'ok'},
 {'loss': -296.0, 'status': 'ok'},
 {'loss': -40.0, 'status': 'ok'},
 {'loss': 281.0, 'status': 'ok'},
 {'loss': 64.0, 'status': 'ok'},
 {'loss': 100.0, 'status': 'ok'},
 {'loss': 60.0, 'status': 'ok'},
 {'loss': -39.0, 'status': 'ok'},
 {'loss': 1.0, 'status': 'ok'},
 {'loss': -164.0, 'status': 'ok'},
 {'loss': 21.0, 'status': 'ok'},
 {'loss': -56.0, 'status': 'ok'},
 {'loss': 284.0, 'status': 'ok'},
 {'loss': 176.0, 'status': 'ok'},
 {'loss': -171.0, 'status': 'ok'},
 {'loss': 0.0, 'status': 'ok'}]

In [71]:
trial_val.vals

{'x': [np.float64(-6.0),
  np.float64(-4.0),
  np.float64(4.0),
  np.float64(-4.0),
  np.float64(9.0),
  np.float64(2.0),
  np.float64(10.0),
  np.float64(-9.0),
  np.float64(-8.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(1.0),
  np.float64(9.0),
  np.float64(6.0),
  np.float64(9.0),
  np.float64(2.0),
  np.float64(-2.0),
  np.float64(-4.0),
  np.float64(7.0),
  np.float64(-0.0)],
 'y': [np.float64(5.0),
  np.float64(10.0),
  np.float64(-2.0),
  np.float64(12.0),
  np.float64(1.0),
  np.float64(15.0),
  np.float64(7.0),
  np.float64(-10.0),
  np.float64(0.0),
  np.float64(-5.0),
  np.float64(-3.0),
  np.float64(2.0),
  np.float64(4.0),
  np.float64(10.0),
  np.float64(3.0),
  np.float64(3.0),
  np.float64(-14.0),
  np.float64(-8.0),
  np.float64(11.0),
  np.float64(-0.0)]}

In [72]:
import pandas as pd

losses = [loss_dict['loss'] for loss_dict in trial_val.results]

# DataFrame 으로 생성
result_df = pd.DataFrame({'x': trial_val.vals['x'],
              'y' : trial_val.vals['y'],
              'losses':losses})

result_df

,x,y,losses
0,-6.0,5.0,-64.0
1,-4.0,10.0,-184.0
2,4.0,-2.0,56.0
3,-4.0,12.0,-224.0
4,9.0,1.0,61.0
5,2.0,15.0,-296.0
6,10.0,7.0,-40.0
7,-9.0,-10.0,281.0
8,-8.0,0.0,64.0
9,-0.0,-5.0,100.0


### HyperOpt를 XGBoost 하이퍼 파라미터 튜닝에 적용

In [73]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
cancer_df['target'] = dataset.target

X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, [-1]]

# 전체 데이터 중 80% 학습용 데이터, 20% 테스트 데이터용 데이터 추출
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, test_size=0.2, random_state=156)

# 학습 데이터를 다시 학습과 검증 데이터로 분리
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=156)

In [74]:
from hyperopt import hp

xgb_search_space = {'max_depth' : hp.quniform('max_depth',5, 20, 1),
                    'min_child_weight' : hp.quniform('min_child_weight', 1, 2, 1),
                    'learning_rate' : hp.uniform('learning_rate', 0.01, 0.2),
                    'colsample_bytree' : hp.uniform('colsample_bytree', 0.5, 1)}

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

# fmin()에서 입력된 search_space값으로 입력된 모든 값은 실수형임
# XGBClassifier의 정수형 하이퍼 파라미터는 정수형 변환을 해줘야 함.
# 정확도는 높을 수록 더 좋은 수치임 -1 * 정확도를 곱해서 큰 정확도 값일 수록 최소가 되도록 반환

def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring="accuracy", cv=5)
    
    # accuracy는 cv=3 개수만큼의 정확도 결과를 가지므로 이를 평균해서 반환하되 -1을 곱해줌 (분류일 때만)
    return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK}

In [76]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(fn=objective_func, space=xgb_search_space, algo=tpe.suggest, max_evals=50,
            trials=trial_val, rstate=np.random.default_rng(seed=9))
print('best', best)

100%|██████████| 50/50 [00:26<00:00,  1.87trial/s, best loss: -0.964835164835165] 
best {'colsample_bytree': np.float64(0.685216903267962), 'learning_rate': np.float64(0.13468864499466687), 'max_depth': np.float64(17.0), 'min_child_weight': np.float64(1.0)}


In [ ]:
print('colsample_bytree {0}, learning_rate {1}, min_child_weight {2}, max_depth {3}'
      .format(best['colsample_bytree'], best['learning_rate'], best['min_child_weight'], best['max_depth']))

colsample_bytree 0.685216903267962, learning_rate 0.13468864499466687, min_child_weight 1.0, max_depth 17.0


In [78]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_curve, roc_curve

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
          F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [86]:
evals = [(X_tr, y_tr),(X_val, y_val)]
xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=round(best['learning_rate'],5),
                            colsample_bytree=round(best['colsample_bytree'],5),
                            min_child_weight=int(best['min_child_weight']),
                            max_depth=int(best['max_depth']),
                            early_stopping_rounds=50,
                            eval_metric='logloss'
                            )

xgb_wrapper.fit(X_tr, y_tr,  eval_set=evals, verbose=True)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

get_clf_eval(y_test, preds, pred_proba)

[0]	validation_0-logloss:0.55791	validation_1-logloss:0.57735


[1]	validation_0-logloss:0.47408	validation_1-logloss:0.51882
[2]	validation_0-logloss:0.40847	validation_1-logloss:0.47189
[3]	validation_0-logloss:0.35371	validation_1-logloss:0.43141
[4]	validation_0-logloss:0.30746	validation_1-logloss:0.39752
[5]	validation_0-logloss:0.27066	validation_1-logloss:0.37316
[6]	validation_0-logloss:0.23875	validation_1-logloss:0.35134
[7]	validation_0-logloss:0.21109	validation_1-logloss:0.33269
[8]	validation_0-logloss:0.18774	validation_1-logloss:0.31896
[9]	validation_0-logloss:0.16705	validation_1-logloss:0.30465
[10]	validation_0-logloss:0.15032	validation_1-logloss:0.29394
[11]	validation_0-logloss:0.13532	validation_1-logloss:0.28505
[12]	validation_0-logloss:0.12227	validation_1-logloss:0.27901
[13]	validation_0-logloss:0.11091	validation_1-logloss:0.27123
[14]	validation_0-logloss:0.10088	validation_1-logloss:0.26235
[15]	validation_0-logloss:0.09244	validation_1-logloss:0.25953
[16]	validation_0-logloss:0.08432	validation_1-logloss:0.25148
[

In [87]:
trial_val.results

[{'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.956043956043956, 'status': 'ok'},
 {'loss': -0.9626373626373625, 'status': 'ok'},
 {'loss': -0.956043956043956, 'status': 'ok'},
 {'loss': -0.9538461538461538, 'status': 'ok'},
 {'loss': -0.9604395604395604, 'status': 'ok'},
 {'loss': -0.956043956043956, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.9604395604395604, 'status': 'ok'},
 {'loss': -0.9538461538461538, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.956043956043956, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.9516483516483516, 'status': 'ok'},
 {'loss': -0.9582417582417582, 'status': 'ok'},
 {'loss': -0.9604395604395604, 'status': 'ok'},
 {'loss': -0.9604395604395604, 'status': 'ok'},
 {'loss': -0.9626373626373625, 'status': 'ok

In [88]:
trial_val.vals

{'colsample_bytree': [np.float64(0.5852347138193622),
  np.float64(0.7271863641855161),
  np.float64(0.9599446282177103),
  np.float64(0.9500116932342133),
  np.float64(0.6743364060621724),
  np.float64(0.8637740285716389),
  np.float64(0.9575208672481308),
  np.float64(0.6950178418953206),
  np.float64(0.684441779397407),
  np.float64(0.5921156978948783),
  np.float64(0.6147984356971752),
  np.float64(0.7767383218403126),
  np.float64(0.5147724605153474),
  np.float64(0.949782533600956),
  np.float64(0.9261209894715933),
  np.float64(0.5709901136927109),
  np.float64(0.8845494605701275),
  np.float64(0.548301545497125),
  np.float64(0.9102783821000844),
  np.float64(0.5325012994457242),
  np.float64(0.6448903552593086),
  np.float64(0.6433903661811702),
  np.float64(0.7804637645307853),
  np.float64(0.646364433721413),
  np.float64(0.8007380873190837),
  np.float64(0.6393263765444664),
  np.float64(0.7350881141705672),
  np.float64(0.685216903267962),
  np.float64(0.5824100781036491),

In [ ]:
print('colsample_bytree {0}, learning_rate {1}, min_child_weight {2}, max_depth {3}'
      .format(best['colsample_bytree'], best['learning_rate'], best['min_child_weight'], best['max_depth']))

In [93]:
losses = [loss_dict['loss'] for loss_dict in trial_val.results]
result_df = pd.DataFrame({'colsample_bytree' : trial_val.vals['colsample_bytree'], 
                          'learning_rate': trial_val.vals['learning_rate'],
                          'min_child_weight' : trial_val.vals['min_child_weight'],
                          'max_depth': trial_val.vals['max_depth'],
                          'losses':losses})
result_df

,colsample_bytree,learning_rate,min_child_weight,max_depth,losses
0,0.585235,0.033688,2.0,19.0,-0.958242
1,0.727186,0.105956,2.0,5.0,-0.958242
2,0.959945,0.154804,2.0,6.0,-0.958242
3,0.950012,0.120686,2.0,6.0,-0.956044
4,0.674336,0.142392,2.0,16.0,-0.962637
5,0.863774,0.106579,2.0,8.0,-0.956044
6,0.957521,0.079111,2.0,14.0,-0.953846
7,0.695018,0.095213,2.0,19.0,-0.960440
8,0.684442,0.147520,2.0,9.0,-0.956044
9,0.592116,0.081179,1.0,8.0,-0.958242


In [96]:
result_df[result_df['losses'] == result_df['losses'].min()]

,colsample_bytree,learning_rate,min_child_weight,max_depth,losses
27,0.685217,0.134689,1.0,17.0,-0.964835
